# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets by their @id and name
print('Available record sets:')
for rs in metadata.record_sets:
    print(f"  - @id: {rs.id}, name: {rs.name}")
    # List the available fields for each record set
    print("    Fields (columns):")
    for field in rs.fields:
        print(f"      - @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.
We will extract data from all available record sets into individual Pandas DataFrames.

In [ ]:
# Gather all record set @ids
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded: {len(df)} records, columns: {df.columns.tolist()}")

# We'll use the first record set for demonstration
main_record_set_id = record_sets[0]
main_df = dataframes[main_record_set_id]
print(f'Columns in Main Record Set (@id = {main_record_set_id}):')
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates filtering proteins by molecular weight, normalizing, and grouping by description.

In [ ]:
# Choose a numeric field for demonstration (e.g., Molecular Weight or 'MW')
# Find a likely numeric field in the record set fields
main_fields = {field.id: field for field in metadata.record_sets[0].fields}
numeric_field_id = None
for field_id, field in main_fields.items():
    if field.data_type in ['Float', 'Integer', 'Number'] or 'weight' in field.name.lower() or 'mw' in field.name.lower():
        numeric_field_id = field_id
        print(f"Using numeric field: @id={numeric_field_id}, name={field.name}")
        break
if not numeric_field_id:
    raise ValueError("No suitable numeric field found.")

# Filter records with the chosen numeric field above a threshold
threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].dtype.kind in 'iufc' else 10
filtered_df = main_df[main_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (mean): {len(filtered_df)} rows")
display_columns = [numeric_field_id]
print(filtered_df[display_columns].head())

# Normalize the numeric field (standard score)
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a string/categorical field if available (e.g., description or accession)
group_field_id = None
for field_id, field in main_fields.items():
    if field.data_type == 'Text' and field_id != numeric_field_id:
        group_field_id = field_id
        print(f"Grouping by field: @id={group_field_id}, name={field.name}")
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f'Distribution of {numeric_field_id}')
plt.show()

if f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True, color='orange')
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.title(f'Normalized Distribution of {numeric_field_id}')
    plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the FAIR<sup>2</sup> dataset 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' with `mlcroissant`.
- We inspected the dataset metadata and identified available record sets and fields via their `@id`s.
- We extracted the main record set to a DataFrame, filtered by a numeric variable, applied normalization, and grouped by a descriptive field.
- Finally, we visualized the field distributions, demonstrating how to use the Croissant schema for reproducible, FAIR dataset analysis.

For deeper analysis, consult each field `@id`'s documentation and the Croissant schema for semantics and units.